In [3]:
import pandas as pd
import os
import re
from dotenv import load_dotenv
from datetime import datetime

load_dotenv()


bronze_dir = os.getenv("BRONZE")

cdate = datetime.now().strftime("%Y%m%d")
# 1. Path Configuration
excel_path = os.path.join(bronze_dir, f"{cdate}.xlsx")


if not os.path.exists(excel_path):
    raise FileNotFoundError(f"Could not find the Excel file at: {excel_path}")

# Load the file
df = pd.read_excel(excel_path)

# Clean up column names to prevent trailing/leading space KeyErrors
df.columns = df.columns.str.strip()

# Target column name variable (adjust if it's lowercase 'd' in your file)
target_col = 'Proposal Details' 

if target_col not in df.columns:
    raise KeyError(f"Could not find '{target_col}' column. Available columns are: {list(df.columns)}")

# 2. Define the Parsing Function
def extract_portal_details(text):
    # Fallback for empty/NaN cells
    if pd.isna(text):
        return pd.Series([None] * 5)
    
    text_str = str(text)
    
    # Using regex lookarounds to capture data between the known field labels
    clearance = re.search(r'Clearance Type:\s*(.*?)(?=\s*S/W No\.:|$)', text_str, re.IGNORECASE)
    sw_no     = re.search(r'S/W No\.\s*:\s*(.*?)(?=\s*Category:|$)', text_str, re.IGNORECASE)
    category  = re.search(r'Category\s*:\s*(.*?)(?=\s*Sector:|$)', text_str, re.IGNORECASE)
    sector    = re.search(r'Sector\s*:\s*(.*?)(?=\s*Date of Submission:|$)', text_str, re.IGNORECASE)
    date_sub  = re.search(r'Date of Submission\s*:\s*(.*?)$', text_str, re.IGNORECASE)
    
    # Extract match if found, strip trailing spaces, otherwise return None
    return pd.Series([
        clearance.group(1).strip() if clearance else None,
        sw_no.group(1).strip() if sw_no else None,
        category.group(1).strip() if category else None,
        sector.group(1).strip() if sector else None,
        date_sub.group(1).strip() if date_sub else None
    ])

# 3. Apply parsing to generate 5 new columns
print("Extracting data points from Proposal Details...")

new_cols = ['Clearance Type', 'S/W No.', 'Category', 'Sector', 'Date of Submission']
df[new_cols] = df[target_col].apply(extract_portal_details)

# 4. Save back to Excel
df.to_excel(excel_path, index=False)
print(f"Success! Extracted fields saved into columns: {new_cols}")

Extracting data points from Proposal Details...
Success! Extracted fields saved into columns: ['Clearance Type', 'S/W No.', 'Category', 'Sector', 'Date of Submission']


In [5]:
#Fill Missing values for Sector
bronze_dir = os.getenv("BRONZE")

cdate = datetime.now().strftime("%Y%m%d")

# Define your file paths
input_path = (
    os.path.join(bronze_dir, f"{cdate}.xlsx")
)
output_path = os.path.join(bronze_dir, f"{cdate}.xlsx")

# 1. Load the Excel file
df = pd.read_excel(input_path)

# Optional: Strip leading/trailing whitespaces to ensure exact matching
df["Activity Description"] = df["Activity Description"].astype(str).str.strip()

# 2. Build the lookup mapping from non-null Sector values
sector_lookup = (
    df.dropna(subset=["Sector"])
    .drop_duplicates(subset=["Activity Description"])
    .set_index("Activity Description")["Sector"]
    .to_dict()
)

# 3. Fill in missing Sector values using the Activity Description lookup
df["Sector"] = df["Sector"].fillna(df["Activity Description"].map(sector_lookup))

# 4. Save to a new Excel file
df.to_excel(output_path, index=False)

print(f"Processing complete! Saved updated file to:\n{output_path}")

Processing complete! Saved updated file to:
F:\Chimney Work\Marketing\Parivesh Work\Data Architecture\Bronze\20260730.xlsx


In [8]:
#Extract Form #
import os
import re
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

cdate = datetime.now().strftime("%Y%m%d")
bronze_dir = os.getenv("BRONZE")
# Define path to target Excel file
file_path = os.path.join(bronze_dir, f"{cdate}.xlsx")

if not os.path.exists(file_path):
    print(f"❌ File not found at path: {file_path}")
else:
    print(f"📂 Loading file: {file_path}")
    df = pd.read_excel(file_path)

    # 1. Strip non-printable ASCII control characters to avoid IllegalCharacterError
    illegal_xml_chars_re = re.compile(r"[\x00-\x08\x0B-\x0C\x0E-\x1F]")
    df = df.map(
        lambda x: illegal_xml_chars_re.sub("", x) if isinstance(x, str) else x
    )

    # 2. Function to extract Form No from the end of string
    def extract_form_number(text):
        if pd.isna(text):
            return None
        match = re.search(
            r"Form\s*[-–]?\s*([A-Za-z0-9]+)\s*$", str(text), re.IGNORECASE
        )
        return f"Form-{match.group(1)}" if match else None

    # Identify the clearance column name dynamically (handles variations)
    clearance_col = None
    for col in df.columns:
        if "clearance" in str(col).lower():
            clearance_col = col
            break

    if clearance_col:
        print(f"🔍 Found target column: '{clearance_col}'")

        # Create new Form_No column right after the clearance type column
        col_idx = df.columns.get_loc(clearance_col) + 1
        form_series = df[clearance_col].apply(extract_form_number)
        df.insert(col_idx, "Form_No", form_series)

        # 3. Save updates back to the same Excel file
        df.to_excel(file_path, index=False)
        print(f"\n✅ File updated successfully at:\n   {file_path}")
        print("\n--- Preview of updated columns ---")
        print(df[[clearance_col, "Form_No"]].head(10))
    else:
        print(
            "❌ Could not locate a 'Clearance Type' column in the Excel file."
        )

📂 Loading file: F:\Chimney Work\Marketing\Parivesh Work\Data Architecture\Bronze\20260730.xlsx
🔍 Found target column: 'Clearance Type'

✅ File updated successfully at:
   F:\Chimney Work\Marketing\Parivesh Work\Data Architecture\Bronze\20260730.xlsx

--- Preview of updated columns ---
                                      Clearance Type  Form_No
0  Application for No Increase in Pollution Load ...  Form-10
1  Application for ToR (Category A, B1, and B2 Vi...   Form-1
2  Application for EC (Category A, B1, and B2 Vio...   Form-1
3            Application for Amendment in EC- Form-4   Form-4
4  Application for EC (Category A, B1, and B2 Vio...   Form-1
5                Application for Corrigendum Form-13  Form-13
6  Application for EC (Category A, B1, and B2 Vio...   Form-1
7  Application for EC (Category A, B1, and B2 Vio...   Form-1
8  Application for EC (Category A, B1, and B2 Vio...   Form-1
9  Application for No Increase in Pollution Load ...  Form-10


In [12]:
from datetime import datetime
import os
import pandas as pd

from dotenv import load_dotenv

load_dotenv()
# 1. Setup paths and dates from environment variables with fallback/validation
cdate = datetime.now().strftime("%Y%m%d")
bronze_dir = os.getenv("BRONZE")
master_db_path = os.getenv("MDB")
log_dir = os.getenv("BRONZELOG")

# Validate environment variables to prevent TypeError
if not bronze_dir or not master_db_path or not log_dir:
  raise EnvironmentError(
      "One or more required environment variables (BRONZE, MDB, BRONZELOG) are not set."
  )

# Target daily pulled file path
file_path = os.path.join(bronze_dir, f"{cdate}.xlsx")

# Ensure log directory exists
os.makedirs(log_dir, exist_ok=True)
log_file_path = os.path.join(log_dir, "processing_log.xlsx")

# 2. Read Master Database and Daily Pulled File
df_master = pd.read_excel(master_db_path)
df_daily = pd.read_excel(file_path)

# Ensure 'Comment' column exists in master DB; if not, initialize existing rows as 'initial records'
if "Comment" not in df_master.columns:
  df_master["Comment"] = "initial records"
else:
  df_master["Comment"] = df_master["Comment"].fillna("initial records")

# 3. Match columns: retain only the columns present in the Master DB (excluding 'Comment' for matching daily data)
master_columns = [col for col in df_master.columns if col != "Comment"]

# Filter daily dataframe to only include columns that exist in the master database
df_daily_filtered = df_daily[[col for col in master_columns if col in df_daily.columns]]

# 4. Identify new proposals by checking against the master database
existing_proposals = set(df_master["Proposal No."])
df_new = df_daily_filtered[~df_daily_filtered["Proposal No."].isin(existing_proposals)].copy()

num_new_records = len(df_new)

# 5. Add comment for new entries and append to master DB
if num_new_records > 0:
  df_new["Comment"] = cdate

  # Ensure columns align perfectly before concatenating
  df_updated_master = pd.concat([df_master, df_new], ignore_index=True)
else:
  df_updated_master = df_master

# Save updated master database back to excel
df_updated_master.to_excel(master_db_path, index=False)

# 6. Create or update the log entry in BRONZELOG
log_entry = pd.DataFrame(
    [{"Date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "New Records Added": num_new_records}]
)

if os.path.exists(log_file_path):
  df_log = pd.read_excel(log_file_path)
  df_log = pd.concat([df_log, log_entry], ignore_index=True)
else:
  df_log = log_entry

df_log.to_excel(log_file_path, index=False)

print(f"Process completed successfully. {num_new_records} new records added.")

Process completed successfully. 555 new records added.


In [ ]:
#Apply Filters and variables
import os
import pandas as pd
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

cdate = datetime.now().strftime("%Y%m%d")


# Define input and output paths
input_path = os.getenv("BRONZE") + f"{cdate}.xlsx")
output_path = os.getenv("BRONZE") + r"\RAW_MERGED - IND2.xlsx"

# 1. Load Excel dataset
df = pd.read_excel(input_path)

# Ensure 'Date of Submission' is treated as datetime for accurate filtering
# (handles mixed string/datetime inputs safely)
df["Date_Parsed"] = pd.to_datetime(
    df["Date of Submission"], format="%d/%m/%Y", errors="coerce"
)

# 2. Build Filter Conditions

# Filter 1: Proposal Status
status_filter = df["Proposal Status"] == "EC Granted"

# Filter 2: Clearance Type
clearance_types = [
    "Application for EC (Category A, B1, and B2 Violation)- Form 1",
    "Application for ToR (Category A, B1, and B2 Violation)/EC (Category B2) - Form 1",
]
clearance_filter = df["Clearance Type"].isin(clearance_types)

# Filter 3: Sector
sectors = [
    "Industrial Projects - 1",
    "Industrial Projects - 2",
    "Industrial Projects - 3",
    "Non-Coal Mining",
    "Thermal Projects",
]
sector_filter = df["Sector"].isin(sectors)

# Filter 4: Date of Submission (Year 2025 or 2026)
year_filter = df["Date_Parsed"].dt.year.isin([2025, 2026])

# Filter 5: Exclude specific Activity Descriptions
excluded_activities = [
    "5(g) Distilleries",
    "5(ga) Grain based distilleries",
    "5(j) Sugar Industry",
]
activity_filter = ~df["Activity Description"].isin(excluded_activities)

# Combine all filters
combined_filter = (
    status_filter
    & clearance_filter
    & sector_filter
    & year_filter
    & activity_filter
)

# 3. Apply Filters
filtered_df = df[combined_filter].copy()

# 4. Keep specified columns only
columns_to_keep = [
    "Proposal No.",
    "Project Name",
    "Location",
    "Project Proponent",
    "Proposal Status",
    "Activity Description",
    "Clearance Type",
    "Sector",
    "Date of Submission",
]

filtered_df = filtered_df[columns_to_keep]

# 5. Save the filtered dataset to the output path
filtered_df.to_excel(output_path, index=False)

print(f"File successfully filtered and saved to: {output_path}")

File successfully filtered and saved to: F:\Chimney Work\Marketing\Parivesh Work\Data Architecture\Bronze\RAW_MERGED - IND2.xlsx


In [ ]:
#work in progress.

import os
import re
import pandas as pd
from dotenv import load_dotenv

load_dotenv()


# Define Taxonomy Dictionary (Main Headings without numbering)
TAXONOMY = {
    "Metallic Minerals": {
        "Iron Ore": [
            "Iron Ore",
            "Hematite",
            "Magnetite",
            "Siderite",
            "Goethite",
            "Iron Sand",
        ],
        "Bauxite": [
            "Bauxite",
            "Aluminium Ore",
            "Alumina Ore",
            "Hydrous Aluminum Oxide",
        ],
        "Copper Ore": [
            "Copper Ore",
            "Copper",
            "Chalcopyrite",
            "Bornite",
            "Malachite",
            "Cuprite",
        ],
        "Manganese Ore": [
            "Manganese Ore",
            "Manganese",
            "Pyrolusite",
            "Psilomelane",
            "Rhodochrosite",
        ],
        "Chromite": ["Chromite", "Chromium Ore", "Ferrochrome Ore"],
        "Gold": [
            "Gold",
            "Native Gold",
            "Gold Ore",
            "Auriferous Quartz",
            "Placer Gold",
        ],
        "Silver": ["Silver", "Native Silver", "Argentite", "Galvanic Silver"],
        "Lead & Zinc": [
            "Lead & Zinc",
            "Lead",
            "Zinc",
            "Galena",
            "Sphalerite",
            "Zinc Blende",
            "Calamine",
        ],
        "Nickel": ["Nickel", "Lateritic Nickel", "Pentandite"],
        "Titanium Ores": [
            "Titanium Ores",
            "Titanium",
            "Ilmenite",
            "Rutile",
            "Titaniferous Magnetite",
        ],
        "Tin": ["Tin", "Cassiterite", "Tinstone"],
    },
    "Energy & Strategic / Atomic Minerals": {
        "Coal": [
            "Coal",
            "Thermal Coal",
            "Coking Coal",
            "Bituminous",
            "Lignite",
            "Brown Coal",
            "Anthracite",
        ],
        "Petroleum & Natural Gas": [
            "Petroleum & Natural Gas",
            "Petroleum",
            "Crude Oil",
            "Mineral Oil",
            "Hydrocarbons",
            "Natural Gas",
            "Shale Gas",
        ],
        "Uranium": ["Uranium", "Pitchblende", "Uraninite"],
        "Thorium": ["Thorium", "Monazite Sand", "Thorianite"],
        "Lithium": ["Lithium", "Spodumene", "Lepidolite", "White Gold"],
    },
    "Non-Metallic & Industrial Minerals": {
        "Limestone": [
            "Limestone",
            "Calcium Carbonate",
            "Calcite",
            "Chalk",
            "Quicklime Stone",
        ],
        "Dolomite": ["Dolomite", "Dolostone", "Magnesium Limestone"],
        "Mica": [
            "Mica",
            "Muscovite",
            "Phlogopite",
            "Biotite",
            "Isinglass",
            "Sheet Mica",
            "Mica Flakes",
        ],
        "Gypsum": ["Gypsum", "Selenite", "Alabaster", "Hydrated Calcium Sulfate"],
        "Rock Phosphate": ["Rock Phosphate", "Phosphorite", "Apatite", "Phosphate Rock"],
        "Fluorspar": ["Fluorspar", "Fluorite", "Calcium Fluoride"],
        "Barite": ["Barite", "Barytes", "Heavy Spar", "Barium Sulfate"],
        "Magnesite": ["Magnesite", "Magnesium Carbonate"],
        "Kyanite & Sillimanite": [
            "Kyanite & Sillimanite",
            "Kyanite",
            "Sillimanite",
            "Aluminosilicate Minerals",
            "Refractory Minerals",
        ],
        "Asbestos": ["Asbestos", "Chrysotile", "Amphibole", "White Asbestos"],
        "Feldspar": ["Feldspar", "Orthoclase", "Plagioclase", "Microcline"],
        "Quartz & Silica": [
            "Quartz & Silica",
            "Silica Sand",
            "Quartzite",
            "Crystal Quartz",
            "Rock Crystal",
        ],
        "Talc & Soapstone": [
            "Talc & Soapstone",
            "Talc",
            "Soapstone",
            "Steatite",
            "French Chalk",
            "Hydrous Magnesium Silicate",
        ],
    },
    "Gemstones & Precious Stones": {
        "Diamond": [
            "Diamond",
            "Carbon Crystal",
            "Gem Diamond",
            "Industrial Diamond",
            "Bort",
        ],
        "Emerald": ["Emerald", "Green Beryl"],
        "Ruby & Sapphire": [
            "Ruby & Sapphire",
            "Ruby",
            "Sapphire",
            "Corundum",
            "Manik",
            "Neelam",
        ],
        "Garnet": ["Garnet", "Almandine", "Pyrope", "Garnet Sand"],
        "Agate & Chalcedony": [
            "Agate & Chalcedony",
            "Agate",
            "Chalcedony",
            "Carnelian",
            "Jasper",
            "Onyx",
            "Silica Stones",
        ],
    },
    "Building & Dimension Stones": {
        "Granite": [
            "Granite",
            "Dimension Stone",
            "Commercial Granite",
            "Black Granite",
        ],
        "Marble": [
            "Marble",
            "Makrana Marble",
            "Calcitic Marble",
            "Dolomitic Marble",
            "Crystalline Limestone",
        ],
        "Sandstone": [
            "Sandstone",
            "Red Fort Stone",
            "Dholpur Stone",
            "Buff Sandstone",
            "Quartzose Sandstone",
        ],
        "Slate & Schist": [
            "Slate & Schist",
            "Slate",
            "Schist",
            "Roofing Slate",
            "Flagstone",
        ],
        "Laterite": ["Laterite", "Laterite Stone", "Building Block Stone"],
        "Basalt & Trap Rock": [
            "Basalt & Trap Rock",
            "Basalt",
            "Trap Rock",
            "Deccan Trap",
            "Black Trap",
            "Crushed Stone",
        ],
    },
    "Sand, Gravel & Aggregates": {
        "River Sand": ["River Sand", "Natural Sand", "Coarse Sand", "Bajri", "Reti"],
        "Silica Sand": ["Silica Sand", "Glass Sand", "Industrial Sand", "Quartz Sand"],
        "Manufactured Sand": [
            "Manufactured Sand",
            "M-Sand",
            "Crushed Rock Sand",
            "Artificial Sand",
        ],
        "Gravel & Aggregate": [
            "Gravel & Aggregate",
            "Gravel",
            "Aggregate",
            "Jit",
            "Gitti",
            "Metal Stones",
            "Crushed Stone Aggregate",
        ],
        "Placer & Beach Sands": [
            "Placer & Beach Sands",
            "Monazite Sand",
            "Heavy Mineral Sand",
            "Black Sand",
        ],
    },
}


def classify_text(text, taxonomy):
    """
    Scans project text for mineral keywords and extracts Main Heading, Subheading, and Keyword.
    If multiple minerals are found, joins them using comma separation.
    """
    if not isinstance(text, str) or not text.strip():
        return "Unclassified", "Unclassified", "None"

    # Flatten taxonomy keywords and sort by keyword length descending (longer phrases match first)
    keyword_map = []
    for main_heading, subheadings in taxonomy.items():
        for subheading, keywords in subheadings.items():
            all_kw = set([subheading] + keywords)
            for kw in all_kw:
                keyword_map.append((kw, main_heading, subheading))

    keyword_map.sort(key=lambda x: len(x[0]), reverse=True)

    matched_main = []
    matched_sub = []
    matched_kw = []
    seen_subheadings = set()

    for kw, main_h, sub_h in keyword_map:
        # Regex search using word boundaries (\b) and case insensitivity
        pattern = r"\b" + re.escape(kw) + r"\b"
        if re.search(pattern, text, flags=re.IGNORECASE):
            if (main_h, sub_h) not in seen_subheadings:
                seen_subheadings.add((main_h, sub_h))
                matched_main.append(main_h)
                matched_sub.append(sub_h)
                matched_kw.append(kw)

    if matched_main:
        # Deduplicate main headings while preserving order
        unique_mains = list(dict.fromkeys(matched_main))
        return (
            ", ".join(unique_mains),
            ", ".join(matched_sub),
            ", ".join(matched_kw),
        )
    else:
        return "Unclassified", "Unclassified", "None"


def update_excel_file(file_path, project_col_name="Project Name"):
    """
    Reads the Excel file, adds/updates classification columns, and saves back to the same path.
    """
    if not os.path.exists(file_path):
        print(f"Error: File not found at path: {file_path}")
        return

    print(f"Loading Excel file: {file_path} ...")
    df = pd.read_excel(file_path)

    # Check if the project column exists (case-insensitive check)
    target_col = None
    for col in df.columns:
        if col.strip().lower() == project_col_name.lower():
            target_col = col
            break

    if not target_col:
        print(
            f"Error: Column '{project_col_name}' not found in Excel sheet. Available columns: {list(df.columns)}"
        )
        return

    print(f"Classifying project names using column '{target_col}' ...")

    # Apply classification row-by-row
    classifications = df[target_col].apply(
        lambda x: classify_text(x, TAXONOMY)
    )

    # Unpack classification results into separate lists
    main_headings = [c[0] for c in classifications]
    subheadings = [c[1] for c in classifications]
    keywords = [c[2] for c in classifications]

    # Add or update the new columns in the DataFrame
    df["Main Heading"] = main_headings
    df["Subheading"] = subheadings
    df["Matched Keyword"] = keywords

    # Save back to the same Excel file
    print(f"Saving updated DataFrame back to: {file_path} ...")
    df.to_excel(file_path, index=False)
    print("Process completed successfully!")


if __name__ == "__main__":
    # Your file path

   

# 1. Path Configuration
    excel_path = os.getenv("BRONZE") + r"\RAW_MERGED.xlsx"
    

    # Pass the path and your project column name if it differs from 'Project Name'
    update_excel_file(excel_path, project_col_name="Project Name")

Loading Excel file: F:\Chimney Work\Marketing\Parivesh Work\Data Architecture\Bronze\RAW_MERGED.xlsx ...
Classifying project names using column 'Project Name' ...
Saving updated DataFrame back to: F:\Chimney Work\Marketing\Parivesh Work\Data Architecture\Bronze\RAW_MERGED.xlsx ...
Process completed successfully!
